# `00_sync_repo` — sync GitHub → Terra workspace

Run **on the Terra Workbench VM** (pmi-ops browser login). No laptop `gsutil`.

## What this notebook does

1. **Checkout** latest `kvg/aou-lr-phase-2` (`main` by default; disposable clone — local edits inside it are discarded)
2. **Stage scripts** → `$WORKSPACE_BUCKET/scripts/`
3. **Stage notebooks** → `$WORKSPACE_BUCKET/notebooks/` and a folder **outside** the clone (default: `./notebooks/` next to the clone)
4. **Stage WDLs** → `$WORKSPACE_BUCKET/wdl/…`
5. **Update method** `FlareByPopulation` in the Methods repo (new snapshot) and **bump** the workspace workflow config to that snapshot (rewrites script File inputs to this bucket)
6. **Upsert** the `flare_lai_exp` data table from `flare/configs/lai_exp.tsv`
7. **Optionally submit** `FlareByPopulation` on incomplete `flare_lai_exp` rows (`SUBMIT_FLARE=True`)

**First time:** upload this notebook into the workspace `edit/` folder (not inside the clone). After sync, open notebooks from `$WORKSPACE_BUCKET/notebooks/` or the local `notebooks/` copy — not from `aou-lr-phase-2/notebooks/`.



## Config


In [ ]:
import os
from pathlib import Path

REPO_URL = os.environ.get("AOU_LR_REPO_URL", "https://github.com/kvg/aou-lr-phase-2.git")
REF = os.environ.get("AOU_LR_REF", "main")
CLONE_DIR = Path(os.environ.get("AOU_LR_REPO_DIR", str(Path.cwd() / "aou-lr-phase-2")))
# Keep notebooks outside the disposable git clone (Jupyter autosave would dirty it).
NOTEBOOK_DEST = Path(os.environ.get("AOU_LR_NOTEBOOK_DEST", str(Path.cwd() / "notebooks")))

# Public repo — leave blank.
GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN") or os.environ.get("GH_TOKEN") or ""

UPSERT_TABLES = ["flare_lai_exp"]

# Push a new FlareByPopulation method snapshot, then point the workspace config at it.
# METHOD_NAMESPACE defaults to the Terra workspace namespace (billing project).
UPDATE_METHODS = [
    {"wdl": "flare/wdl/FlareByPopulation.wdl", "name": "FlareByPopulation"},
]
METHOD_NAMESPACE = os.environ.get("TERRA_METHOD_NAMESPACE", "")  # blank → workspace ns
BUMP_CONFIGS = [
    {"config_name": "FlareByPopulation", "method_name": "FlareByPopulation"},
]

# Workflow launch (expensive). Default off — set True after reviewing incomplete rows.
SUBMIT_FLARE = os.environ.get("AOU_LR_SUBMIT_FLARE", "").lower() in {"1", "true", "yes"}
# None → all incomplete rows (missing anc_vcf / models_tsv). Or list ids explicitly:
SUBMIT_ENTITY_IDS = None  # e.g. ["pin_gen8_afr_amr", "chr20_em_props_defaults"]
SUBMIT_ONLY_INCOMPLETE = True
SUBMIT_CONFIG_NAME = "FlareByPopulation"

DRY_RUN = os.environ.get("AOU_LR_SYNC_DRY_RUN", "").lower() in {"1", "true", "yes"}

print("WORKSPACE_BUCKET:", os.environ.get("WORKSPACE_BUCKET", ""))
print("CLONE_DIR:", CLONE_DIR)
print("NOTEBOOK_DEST:", NOTEBOOK_DEST)
print("REF:", REF)
print("UPDATE_METHODS:", UPDATE_METHODS)
print("BUMP_CONFIGS:", BUMP_CONFIGS)
print("SUBMIT_FLARE:", SUBMIT_FLARE, "ids:", SUBMIT_ENTITY_IDS)
print("DRY_RUN:", DRY_RUN)



## Bootstrap

Clone/pull enough of the repo to import `terra_sync_repo` if it is not already on this VM.


In [ ]:
import subprocess
import sys

def _run(cmd, **kw):
    print("+", " ".join(map(str, cmd)) if isinstance(cmd, list) else cmd)
    return subprocess.run(cmd, check=True, text=True, **kw)

token = GITHUB_TOKEN
url = REPO_URL
auth_url = (
    f"https://x-access-token:{token}@" + url.split("https://", 1)[1]
    if token and url.startswith("https://")
    else url
)

CLONE_DIR.parent.mkdir(parents=True, exist_ok=True)
if CLONE_DIR.exists() and any(CLONE_DIR.iterdir()) and not (CLONE_DIR / ".git").is_dir():
    raise SystemExit(f"{CLONE_DIR} exists and is not a git checkout")

if (CLONE_DIR / ".git").is_dir():
    _run(["git", "remote", "set-url", "origin", auth_url], cwd=str(CLONE_DIR))
    _run(["git", "fetch", "--tags", "--force", "origin"], cwd=str(CLONE_DIR))
    # Disposable mirror: discard Jupyter autosaves inside the clone.
    _run(["git", "reset", "--hard", f"origin/{REF}"], cwd=str(CLONE_DIR))
    _run(["git", "clean", "-fd"], cwd=str(CLONE_DIR))
    _run(["git", "checkout", "-B", REF, f"origin/{REF}"], cwd=str(CLONE_DIR))
else:
    try:
        _run(["git", "clone", "--branch", REF, "--single-branch", auth_url, str(CLONE_DIR)])
    except subprocess.CalledProcessError:
        _run(["git", "clone", auth_url, str(CLONE_DIR)])
        _run(["git", "checkout", REF], cwd=str(CLONE_DIR))
if token:
    _run(["git", "remote", "set-url", "origin", REPO_URL], cwd=str(CLONE_DIR))

scripts = CLONE_DIR / "scripts"
assert (scripts / "terra_sync_repo.py").is_file(), scripts
if str(scripts) not in sys.path:
    sys.path.insert(0, str(scripts))
print("import path:", scripts)
print("HEAD:", _run(["git", "rev-parse", "--short", "HEAD"], cwd=str(CLONE_DIR), capture_output=True).stdout.strip())



## Sync + optional submit


In [ ]:
import json
from terra_sync_repo import sync_all

if not os.environ.get("WORKSPACE_BUCKET") and not DRY_RUN:
    raise SystemExit("WORKSPACE_BUCKET unset — run on a Terra notebook VM (or DRY_RUN=true)")

report = sync_all(
    repo_url=REPO_URL,
    ref=REF,
    clone_dir=CLONE_DIR,
    github_token=GITHUB_TOKEN or None,
    notebook_dest=NOTEBOOK_DEST,
    stage_scripts_flag=True,
    stage_notebooks_flag=True,
    stage_wdls_flag=True,
    upsert_tables=UPSERT_TABLES,
    update_methods=UPDATE_METHODS or None,
    method_namespace=METHOD_NAMESPACE or None,
    bump_configs=BUMP_CONFIGS or None,
    submit_flare=SUBMIT_FLARE,
    submit_entity_ids=SUBMIT_ENTITY_IDS,
    submit_only_incomplete=SUBMIT_ONLY_INCOMPLETE,
    submit_config_name=SUBMIT_CONFIG_NAME,
    dry_run=DRY_RUN,
)
print(json.dumps(report.to_dict(), indent=2))
if report.warnings:
    print("\nWarnings:")
    for w in report.warnings:
        print("-", w)


## After sync

1. Confirm `$WORKSPACE_BUCKET/scripts/` and `$WORKSPACE_BUCKET/notebooks/` look fresh.
2. In Terra **Workflows**, `FlareByPopulation` should show the new method snapshot (if method update succeeded). If Methods-repo write is denied, import from `$WORKSPACE_BUCKET/wdl/flare/wdl/FlareByPopulation.wdl` once.
3. Confirm `flare_lai_exp` matches `flare/configs/lai_exp.tsv`.
4. To launch incomplete rows: set `SUBMIT_FLARE = True` (or `AOU_LR_SUBMIT_FLARE=true`) and re-run the sync cell — or list `SUBMIT_ENTITY_IDS`.
5. Score finished rows with `flare_02_lai_exp_compare.ipynb` (Part 8).

```bash
python3 aou-lr-phase-2/scripts/terra_sync_repo.py \
  --ref main \
  --update-method flare/wdl/FlareByPopulation.wdl \
  --bump-config FlareByPopulation \
  --upsert-tables flare_lai_exp \
  --submit-flare
```
